<a href="https://colab.research.google.com/github/fecheromero/PNL_tps_Romero_Federico/blob/main/Desafio_1_Romero_Federico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palabra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

# 1. Vectorizar documentos
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.



In [ ]:
##funciones auxiliares

def get_first_n_words(text, n):

  words = text.split()


  first_n_words_list = words[:n]

  result = " ".join(first_n_words_list)

  return result


def get_document_topic(doc_index):
  return newsgroups_train.target_names[y_train[doc_index]]

In [ ]:
from tabulate import tabulate

selectedIndexes = [8992, 10254, 5117, 9971, 5154]
matrixForPrint = []

col_headers = ["index", "first 5 words", "topic", "five closer documents index","cossim", "five similar docs topics"]

row_indices = selectedIndexes

for index in selectedIndexes:
  print(f"index: {index}")
  documentAttributes = []

  ##index
  documentAttributes.append(index)

  ##first 5 words
  documentAttributes.append(get_first_n_words(newsgroups_train.data[index],5))

  ##topic
  documentAttributes.append(get_document_topic(index))

  ##5 similar documents
  cossim = cosine_similarity(X_train[index], X_train)[0]
  five_closser = np.argsort(cossim)[::-1][1:6]
  documentAttributes.append(five_closser)

  ##cossim

  documentAttributes.append(list(map(lambda x: float(cossim[x]),five_closser)))

  ## 5 similar documents topics
  five_similar_doc_topics = []
  five_similar_matrix = [];

  for doc_index in five_closser:
    topic = newsgroups_train.target_names[y_train[doc_index]]
    five_similar_doc_topics.append(topic)

    five_similar_matrix.append([doc_index,float(cossim[doc_index]),topic, get_first_n_words(newsgroups_train.data[doc_index],1000)])

  print('-----------------------------------------------------------\n')
  print(f'first 1000 words')
  print(get_first_n_words(newsgroups_train.data[index],1000))
  print('-----------------------------------------------------------\n')

  print(f'Five cossim for index {index}')

  print(tabulate(five_similar_matrix,headers=['index', 'conssim','topic', '1000 words'], showindex=[1,2,3,4,5], tablefmt="grid"))


  documentAttributes.append(five_similar_doc_topics)

  matrixForPrint.append(documentAttributes)


index: 8992
-----------------------------------------------------------

first 1000 words
While not exactly a service incident, I had a similar experience recently when I bought a new truck. I had picked out the vehicle I wanted and after a little haggling we agreed on a price. I wrote them a check for the down payment plus tax and license and told them I'd be back that evening to pick up the truck. When I returned, I had to wait about an hour before the finance guy could get to me. When I finally got in there, everything went smoothly until he started adding up the numbers. He then discovered that they had miscalculated the tax & license by about $150. He then said he needed another $150 from me. I said we had already agreed on a price and it was their problem, I wasn't giving them any more money. The finance guy then brought in the manager on duty who proceeded to give me a hard time. I reminded him that I was the customer and I didn't think I should be treated like that and that if 

In [ ]:
print(tabulate(matrixForPrint,headers=col_headers, showindex=row_indices, tablefmt="grid"))

+-------+---------+----------------------------------+--------------------+---------------------------------+----------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------+
|       |   index | first 5 words                    | topic              | five closer documents index     | cossim                                                                                                   | five similar docs topics                                                                                          |
+=======+=========+==================================+====================+=================================+==========================================================================================================+===================================================================================================================+
|

### Análisis de Similitud de Documentos (Coseno)

*Se ha utilizado IA (Gemini 3.1 Pro) a partir de un boceto de reflexiones para estructurar esta respuesta, corregir redacción y formato. El análisis cualitativo de los resultados es propio, salvo algún extra de reflexión insinuado en la conversación con el agente.*

A continuación, se presenta un análisis cualitativo de la similitud entre los documentos seleccionados y sus respectivos *top-5* resultados arrojados por el modelo.

Este análisis busca identificar la coherencia semántica y de pertenencia al tópico original, además de detectar posibles anomalías en la recuperación.

---

#### 📄 Índice 8992
> **Tópico original:** `rec.autos`
> **Contenido principal:** Relato/denuncia extensa sobre una mala experiencia de compra o servicio al consumidor en una concesionaria automotriz.

*   **Top 1:** Coincide temática y conceptualmente. Es otra queja/denuncia sobre la compra de un automóvil y el servicio al consumidor. El score de similitud (cossim) es superior a `0.5`, lo cual correlaciona directamente con la exactitud de la predicción.
*   **Top 2 al 5:** Se alejan marcadamente del dominio automotriz. Si bien existe un "nexo conector" abstracto (la idea de relatar una queja, testimonio o "denuncia"), los textos recuperados pertenecen en realidad a denuncias sobre violaciones de derechos humanos en Medio Oriente (`talk.politics.mideast`).
*   **Observación Técnica:** El sistema está logrando identificar la dimensión de un relato testimonial o denuncia, pero confunde dominios completamente disjuntos (autos vs. política internacional). Además, la presencia reiterativa de los mismos documentos de Medio Oriente sugiere un problema de **efecto atractor por longitud**, donde documentos muy largos acumulan coincidencias espurias de *stop words* o vocabulario común.

---

#### 📄 Índice 10254
> **Tópico original:** `alt.atheism`
> **Contenido principal:** Comentario breve y algo ambiguo haciendo una observación personal sobre Alemania y el concepto de un "lema" (*motto*).

*   **Scores generales:** Todos los valores de similitud coseno se mantienen bajos (por debajo de `0.5`), reflejando falta de solidez en las recuperaciones.
*   **Top 1, 2 y 5:** Pertenecen al mismo tópico de ateísmo y a debates de religión. Son textos relativamente cortos enlazados por el uso de términos clave como "Germany/Germanic" y "motto".
*   **Top 3 y 4:** Nuevamente irrumpen los textos largos pertenecientes a las denuncias de lesa humanidad en Medio Oriente (`talk.politics.mideast`). Pierden total completitud temática.
*   **Observación Técnica:** Es altamente probable que estos documentos extensos desplacen a otros más pertinentes simplemente por su volumen.

---

#### 📄 Índice 5117 (y relacionado al análisis 9971)
> **Tópico original (5117):** Discusión/debate sobre Cristianismo (`talk.religion.misc`).

*   **Scores generales:** Se mantienen por debajo de `0.5`.
*   **Top 1, 3 y 4:** Coinciden sólidamente en la temática principal (cristianismo e intercambios coloquiales con cierto nivel de hostilidad/informalidad).
*   **Top 2:** Coincide en la macro-temática (religión) pero es semántica y estructuralmente distinto (parece el fragmento de un sermón formal y pertenece a `soc.religion.christian`).
*   **Top 5:** Perteneciente a `alt.atheism`. Es un falso positivo léxico: comparte el campo semántico y el uso de terminología cristiana, pero desde la postura opuesta o de debate.

---

#### 📄 Índice 9971
> **Tópico original:** Opinión personal sobre la legalización de drogas. Texto extremadamente corto.

*   **Scores generales:** Muy bajos (menores a `0.25`).
*   **Top 1 y 2:** Recuperación exitosa. Pertenecen a foros de debate sobre la DEA, la lucha contra las drogas y su legalización.
*   **Top 3, 4 y 5:** Falsos positivos por coincidencia léxica parcial. El **Top 3** discute la legalización del aborto en Alemania. El **Top 4** es una extensa reflexión sobre cristianismo (de nuevo, un documento anómalamente largo o fuera de lugar que captura similitud residual). El **Top 5** es un discurso político que apenas menciona las drogas al pasar en relación con los jóvenes.
*   **Observación Técnica:** Al ser un documento origen extremadamente corto, la representación vectorial apenas tiene "dirección", permitiendo que cualquier texto que accidentalmente posea algunas de esas palabras cruce el umbral mínimo de similitud.

---

#### 📄 Análisis del Índice 5154
> **Tópico original:** `comp.graphics`
> **Contenido principal:** Comentario técnico muy breve sobre algo técnico, podría ser hardware, mencionando que la "68070 es solo una 68010 con una MMU" y una firma.

*   **Scores generales:** Bajos (entre `0.11` y `0.33`). Esto es esperado al tratarse de un texto origen muy breve (apenas dos oraciones).
*   **Top 1 al 4:** Pertenecen al tópico original (`comp.graphics`). Los resultados muestran una recuperación semántica sorprendentemente precisa dadas las circunstancias.
    *   El **Top 1** recupera un post *del mismo autor*. Probablemente gran parte de la coincidencia surja de la firma.
    *   Del **Top 2 al 4** recuperan discusiones técnicas específicas sobre estos dispositivos nombrados numéricamente **68070, 68010**. Claramente la coincidencia se da por la repetición del número.
*   **Top 5:** Perteneciente al tópico `rec.motorcycles`. Acá vemos un **falso positivo por ambigüedad léxica**. El modelo posiblemente haya matcheado la palabra "Moto" que no termina de entender su significado en 5154, pero seguramente no esté relacionado con el uso de "moto" en contexto de motorcycles.

***
**Reflexión integral:**
El comportamiento global evaluado muestra dos fallas clásicas en sistemas puramente TF-IDF:
1.  **El Pozo Semántico (Efecto atractor por longitud):** Documentos extremadamente largos saturan el espacio vectorial de palabras, capturando falsamente a documentos muy cortos.
2.  **Atadura Léxica y Polisemia:** Falsos positivos por palabras que se utilizan con significado diverso pero se escriben igual.

# 2. Construir un modelo de clasificación por prototipos (tipo zero-shot).

Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

In [ ]:
y_pred_zero_shot = []

# Define the size of the test subset
subset_size = 1000

# Take a subset of X_test and y_test
X_test_subset = X_test[:subset_size]
y_test_subset = y_test[:subset_size]

for i in range(X_test_subset.shape[0]):
    # Get the current test document vector
    test_doc_vector = X_test_subset[i]

    # Calculate cosine similarity with all training documents
    similarities = cosine_similarity(test_doc_vector, X_train)[0]

    # Find the index of the most similar training document
    most_similar_train_idx = np.argmax(similarities)

    # Assign the class of the most similar training document
    predicted_class = y_train[most_similar_train_idx]
    y_pred_zero_shot.append(predicted_class)

# Convert the predictions list to a numpy array for F1-score calculation
y_pred_zero_shot = np.array(y_pred_zero_shot)

# Evaluate the model using the subset of y_test
f1_macro_zero_shot = f1_score(y_test_subset, y_pred_zero_shot, average='macro')

print(f"F1-Score Macro for Zero-Shot Prototype Model (subset of {subset_size} documents): {f1_macro_zero_shot}")

F1-Score Macro for Zero-Shot Prototype Model (subset of 6000 documents): 0.5096765917075549


## Conclusión de la Clasificación Zero-Shot

Esta clasificación manual, que utiliza el *label* del documento con mayor similitud coseno (reutilizando la vectorización TF-IDF), mostró un F1-score macro entre `0.49` y `0.5`. Este resultado representa casi un `10%` menos de efectividad en comparación con la clasificación Naïve Bayes utilizada en la sección inicial de este Colab.

Esta pérdida de efectividad se debe a que la clasificación no se basa en un modelo que aprenda las características del vocabulario para cada clase, sino que simplemente copia la clasificación real del documento más similar. Si la similaridad entre los documentos no es lo suficientemente precisa (como hemos analizado en el punto anterior, donde se encontraron casos de similitud semántica débil o errónea), esto puede generar un aumento en la cantidad de clasificaciones erróneas. De hecho, en la muestra de 5 documentos analizados previamente, si bien el top 1 de similitud a menudo compartía la misma categoría que el documento objetivo, encontramos inconsistencias a partir del top 2. Es muy probable que, al tomar una muestra más grande en ese análisis, se detecten inconsistencias entre la categoría del documento objetivo y su top 1 de similitud.

# 3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.



In [ ]:
## Creando array de vectorizadores para analizar distintas configuraciónes

class Vectorizer:
    def __init__(self, vectorizer_instance, description):
        self.vectorizer = vectorizer_instance
        self.description = description

# Instantiate TfidfVectorizers with different parameters
vectorizers_list = [Vectorizer(tfidfvect, 'Original TfidfVectorizer (default)')]

# Configuration 1
params1 = {'min_df': 1, 'max_df': 0.7}
tfidf_vect1 = TfidfVectorizer(**params1)
description1 = f"TfidfVectorizer with min_df: {params1['min_df']} and max_df: {params1['max_df']}"
vectorizers_list.append(Vectorizer(tfidf_vect1, description1))

# Configuration 2
params2 = {'min_df': 5, 'max_df': 0.5}
tfidf_vect2 = TfidfVectorizer(**params2)
description2 = f"TfidfVectorizer with min_df: {params2['min_df']} and max_df: {params2['max_df']}"
vectorizers_list.append(Vectorizer(tfidf_vect2, description2))

# Configuration 3
params3 = {'min_df': 0.1, 'max_df': 0.4}
tfidf_vect3 = TfidfVectorizer(**params3)
description3 = f"TfidfVectorizer with min_df: {params3['min_df']} and max_df: {params3['max_df']}"
vectorizers_list.append(Vectorizer(tfidf_vect3, description3))

# Configuration 4
params4 = {'min_df': 3, 'max_df': 1.0}
tfidf_vect4 = TfidfVectorizer(**params4)
description4 = f"TfidfVectorizer with min_df: {params4['min_df']} and max_df: {params4['max_df']}"
vectorizers_list.append(Vectorizer(tfidf_vect4, description4))

# Configuration 5
params5 = {'min_df': 2, 'max_df': 0.4}
tfidf_vect5 = TfidfVectorizer(**params5)
description5 = f"TfidfVectorizer with min_df: {params5['min_df']} and max_df: {params5['max_df']}"
vectorizers_list.append(Vectorizer(tfidf_vect5, description5))

# Configuration 5
params5 = {'min_df': 1, 'max_df': 1.0}
tfidf_vect5 = CountVectorizer(**params5)
description5 = f"CountVectorizer with min_df: {params5['min_df']} and max_df: {params5['max_df']}"
vectorizers_list.append(Vectorizer(tfidf_vect5, description5))

# Display the descriptions of the created vectorizers
for i, vec_obj in enumerate(vectorizers_list):
    print(f"Vectorizer {i+1}: {vec_obj.description}")


Vectorizer 1: Original TfidfVectorizer (default)
Vectorizer 2: TfidfVectorizer with min_df: 1 and max_df: 0.7
Vectorizer 3: TfidfVectorizer with min_df: 5 and max_df: 0.5
Vectorizer 4: TfidfVectorizer with min_df: 0.1 and max_df: 0.4
Vectorizer 5: TfidfVectorizer with min_df: 3 and max_df: 1.0
Vectorizer 6: TfidfVectorizer with min_df: 2 and max_df: 0.4
Vectorizer 7: CountVectorizer with min_df: 1 and max_df: 1.0


In [ ]:
# Creando array de modelos para probar distintas configuraciónes y modelos

# 1. Define a class named Models
class Models:
    def __init__(self, model_instance, description):
        self.model = model_instance
        self.description = description

# 2. Create an empty list called models_list
models_list = []

# 3. Instantiate a MultinomialNB model with alpha=0.2
mnb_02 = MultinomialNB(alpha=0.2)
models_list.append(Models(mnb_02, "MultinomialNB with alpha=0.2"))

# 4. Instantiate another MultinomialNB model with alpha=1.0
mnb_10 = MultinomialNB(alpha=1.0)
models_list.append(Models(mnb_10, "MultinomialNB with alpha=1.0"))

# 5. Instantiate a ComplementNB model with alpha=0.2
cnb_02 = ComplementNB(alpha=0.2)
models_list.append(Models(cnb_02, "ComplementNB with alpha=0.2"))

# 6. Instantiate another ComplementNB model with alpha=1.0
cnb_10 = ComplementNB(alpha=1.0)
models_list.append(Models(cnb_10, "ComplementNB with alpha=1.0"))

# 7. Print the descriptions of all models
print("Instantiated Naive Bayes Models:")
for i, model_obj in enumerate(models_list):
    print(f"Model {i+1}: {model_obj.description}")

Instantiated Naive Bayes Models:
Model 1: MultinomialNB with alpha=0.2
Model 2: MultinomialNB with alpha=1.0
Model 3: ComplementNB with alpha=0.2
Model 4: ComplementNB with alpha=1.0


In [ ]:
#Guardando los resultados de entrenar todas las opciones de modelos con todas las opciones de vectorizadores

# Ensure y_train and y_test are defined (if not already from previous cells)
y_train = newsgroups_train.target
y_test = newsgroups_test.target

results_matrix = {}
# Loop through each vectorizer
for vec_obj in vectorizers_list:
    print(f"\nProcessing Vectorizer: {vec_obj.description}")

    # Fit the vectorizer on training data and transform both train and test data
    X_train_vec = vec_obj.vectorizer.fit_transform(newsgroups_train.data)
    X_test_vec = vec_obj.vectorizer.transform(newsgroups_test.data)

    # Initialize an inner dictionary for the current vectorizer if it doesn't exist
    if vec_obj.description not in results_matrix:
        results_matrix[vec_obj.description] = {}

    # Loop through each model
    for model_obj in models_list:
        print(f"  Training and evaluating Model: {model_obj.description}")

        # Train the model
        model_obj.model.fit(X_train_vec, y_train)

        # Predict on the test set
        y_pred = model_obj.model.predict(X_test_vec)

        # Calculate F1-score
        f1 = f1_score(y_test, y_pred, average='macro')

        # Store the F1-score
        results_matrix[vec_obj.description][model_obj.description] = f1

print("\nAll evaluations complete. Results stored in results_matrix.")


Processing Vectorizer: Original TfidfVectorizer (default)
  Training and evaluating Model: MultinomialNB with alpha=0.2
  Training and evaluating Model: MultinomialNB with alpha=1.0
  Training and evaluating Model: ComplementNB with alpha=0.2
  Training and evaluating Model: ComplementNB with alpha=1.0

Processing Vectorizer: TfidfVectorizer with min_df: 1 and max_df: 0.7
  Training and evaluating Model: MultinomialNB with alpha=0.2
  Training and evaluating Model: MultinomialNB with alpha=1.0
  Training and evaluating Model: ComplementNB with alpha=0.2
  Training and evaluating Model: ComplementNB with alpha=1.0

Processing Vectorizer: TfidfVectorizer with min_df: 5 and max_df: 0.5
  Training and evaluating Model: MultinomialNB with alpha=0.2
  Training and evaluating Model: MultinomialNB with alpha=1.0
  Training and evaluating Model: ComplementNB with alpha=0.2
  Training and evaluating Model: ComplementNB with alpha=1.0

Processing Vectorizer: TfidfVectorizer with min_df: 0.1 and 

In [ ]:
#Tabulando resultados para mejor lectura

# Convert results_matrix to a pandas DataFrame for easier tabulation
df_results = pd.DataFrame(results_matrix).T

df_results = df_results[sorted(df_results.columns)]

print(tabulate(df_results, headers='keys', tablefmt='grid', floatfmt=".4f"))

+--------------------------------------------------+-------------------------------+-------------------------------+--------------------------------+--------------------------------+
|                                                  |   ComplementNB with alpha=0.2 |   ComplementNB with alpha=1.0 |   MultinomialNB with alpha=0.2 |   MultinomialNB with alpha=1.0 |
+==================================================+===============================+===============================+================================+================================+
| Original TfidfVectorizer (default)               |                        0.6997 |                        0.6930 |                         0.6425 |                         0.5854 |
+--------------------------------------------------+-------------------------------+-------------------------------+--------------------------------+--------------------------------+
| TfidfVectorizer with min_df: 1 and max_df: 0.7   |                        0.7002 | 

## Conclusiones de los Resultados

### Respecto a los Vectorizadores:
*   El **CountVectorizer** consistentemente arrojó peores resultados, lo cual es esperado dado que no pondera la importancia de las palabras, a diferencia de TF-IDF.
*   Al aumentar el **min_df** por encima de 1 documento, el rendimiento del modelo empeoró en la mayoría de los casos. Notablemente, con un `min_df` del 10% (0.1), el F1-score cayó drásticamente, lo que sugiere que términos que aparecen en pocos documentos (incluso en solo dos) pueden ser cruciales para una clasificación efectiva. Reducir `min_df` para eliminar solo términos que aparecen en un único documento (`min_df=1`) parece ser la estrategia más efectiva.
*   En cuanto al **max_df**, se observó un patrón interesante: reducirlo (por ejemplo, de 1.0 a 0.7) generó una leve mejora en el F1-score en la mayoria de los  modelos. Esto indica que existen *stop words* o términos muy frecuentes que no estan presentes en el 100% de los documentos por lo que no son capturados con el max_df default de 100%. Este comportamiento tiene sentido, ya que los documentos son heterogéneos y varían en longitud.

*Nota: Se decidió no explorar otros parámetros de los vectorizadores, asumiendo que los cambios en `min_df` y `max_df` eran los más relevantes para esta etapa de análisis.*

### Respecto a los Modelos Naïve Bayes:
*   La reducción del parámetro `alpha` (smoothing) de 1.0 a 0.2, tanto en `MultinomialNB` como en `ComplementNB`, generalmente resultó en una mejora significativa del F1-score. Esto sugiere que el vocabulario del dataset es grande y que un smoothing aditivo excesivo (como `alpha=1.0`) puede distorsionar las probabilidades reales de las palabras.
*   El modelo **ComplementNB** superó sistemáticamente a **MultinomialNB** en todos los escenarios probados. Esta es una indicación común de que el dataset de entrenamiento podría estar desbalanceado en cuanto a la distribución de las clases, un escenario para el cual `ComplementNB` está específicamente diseñado para manejar mejor.

### Ganador:
La combinación de **TF-IDF con `min_df=1` y `max_df=0.7`** (la configuración más cercana a `min_df:1 y max_df: 100%`) junto con el modelo **ComplementNB con `alpha=0.2`** logró el F1-score más alto, alcanzando aproximadamente **0.70**. Este resultado es considerablemente superior al 0.585 obtenido con la configuración inicial.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [ ]:

# 1. Transpose X_train to get X_train_T (term-document matrix)
X_train_T = X_train.T


vocabulary_size = X_train_T.shape[0]
# 2. Transpose X_test
X_test_T = X_test.T

# 3. Select 6 random unique word indices
random_word_indices = [76686,28429,73511,50158,38728,65310]
print(f"Shape of X_train_T: {X_train_T.shape}")
print(f"5 random unique word indices: {random_word_indices}")


Shape of X_train_T: (101631, 11314)
5 random unique word indices: [76686, 28429, 73511, 50158, 38728, 65310]


In [ ]:

idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

table_data = []
headers = ["Original Word", "Original Doc Count", "Similar Word", "Similarity", "Similar Doc Count"]

for i, word_idx in enumerate(random_word_indices):
    original_word = idx2word[word_idx]

    # Count documents for the original word
    doc_count_original = X_train_T[word_idx].nnz

    word_vector = X_train_T[word_idx]
    similarities = cosine_similarity(word_vector, X_train_T)[0]
    top_5_similar_indices = np.argsort(similarities)[::-1][1:6] # Exclude itself

    for sim_idx in top_5_similar_indices:
        similar_word = idx2word[sim_idx]
        similarity_score = similarities[sim_idx]

        # Count documents for the similar word
        doc_count_similar = X_train_T[sim_idx].nnz

        table_data.append([original_word, doc_count_original, similar_word, f"{similarity_score:.4f}", doc_count_similar])

print("\n--- Word Similarity Analysis (Detailed) ---")
print(tabulate(table_data, headers=headers, tablefmt="grid"))



--- Word Similarity Analysis (Detailed) ---
+-----------------+----------------------+----------------+--------------+---------------------+
| Original Word   |   Original Doc Count | Similar Word   |   Similarity |   Similar Doc Count |
+=================+======================+================+==============+=====================+
| recommened      |                    1 | worley         |       0.9945 |                   3 |
+-----------------+----------------------+----------------+--------------+---------------------+
| recommened      |                    1 | tracers        |       0.9507 |                   2 |
+-----------------+----------------------+----------------+--------------+---------------------+
| recommened      |                    1 | avid           |       0.3624 |                   6 |
+-----------------+----------------------+----------------+--------------+---------------------+
| recommened      |                    1 | steep          |       0.2824 |        

### Observaciones de la Similitud de Palabras

El análisis de similitud de palabras realizado con la selección de términos arrojó los siguientes resultados:

1.  **Palabra Bonus: 'recommened'**
    *   **Similares:** 'worley', 'tracers', 'avid', 'steep', 'curve'
    *   **Observación:** 'recommened' es probablemente un error tipográfico de 'recommended' y aparece solo en 1 documento. Sus similares mas cercanos tienen muy poca presencia tambien. Deje este caso para analizar que pasa con un typo o una palabra muy rara, el analisis de similitud en este caso considero que debe ser despreciado porque no hay posibilidad de interpretación semantica.

2.  **Palabra: 'colormaps'**
    *   **Similares:** '0xffffffff', 'blue_max', 'cmapalloc', 'base_pixel', 'blue_mult'
    *   **Observación:** Todos los términos similares son altamente técnicos y parecen ser códigos hexadecimales o nombres de variables/funciones relacionadas con la manipulación de colores o gráficos. La similitud de 0.6403 para todos ellos y su aparición en solo 1 documento (excepto 'colormaps' en 8) es muy coherente, indicando que 'colormaps' co-ocurre con estos términos específicos dentro de documentos técnicos sobre gráficos o desarrollo de software.

3.  **Palabra: 'promiscuity'**
    *   **Similares:** 'depression', 'bond', 'hetero', 'pondered', 'relationships'
    *   **Observación:** La mayoría de los términos son semánticamente relacionados con discusiones sobre psicología, sociología y relaciones humanas. 'depression', 'bond', 'hetero', 'relationships' son claramente conceptos asociados a la conducta sexual, relaciones o estados emocionales. 'pondered' como verbo o adjetivo claramente encaja en el contexto de discusiones sobre estos temas.

4.  **Palabra: 'innovations'**
    *   **Similares:** 'innovation', 'developments', 'debatable', 'examined', 'quarters'
    *   **Observación:** 'innovation' es el singular de la palabra original, lo cual es una similitud natural. 'developments' es un sinónimo o término muy cercano. 'debatable' y 'examined' sugieren que las innovaciones son a menudo objeto de discusión y análisis. 'quarters' es menos claro, podría referirse a una medida de tiempo, de todas formas tiene una similitud de 0.4.

5.  **Palabra: 'experiment'**
    *   **Similares:** 'compiliers', 'dyson', 'assemblers', 'toricelli', 'rarety'
    *   **Observación:** Presencia de typos y palabras muy raras. La similitud en este caso es baja (por debajo de 0.3). Es realmente llamativo que una palabra que aparece en 59 documentos tenga su top-1 de similaridad con un typo que aparece solo en 1. Interpretó que el termino se usa en documentos muy variados por lo que no encuentra coincidencias en mas de 1 documento con la misma palabra y esto genera heterogeneidad en sus similitudes.

6.  **Palabra: 'necessity'**
    *   **Similares:** 'tyrants', 'plea', 'infringement', 'slaves', 'creed'
    *   **Observación:** Los términos similares giran en torno a conceptos de opresión, derechos, súplicas y creencias. Claramente hay una ventaja en similaridad de 0.57 con tyrants presente en un par de documentos al menos. Esto sugiere un contexto político, fisolofico o ético del uso del termino.
